### Data Cleaning Process
##### Main Goal: Checking data quality, applying transparent cleaning rules, and then preparing CSV tables for analysis
This notebook does not fill missing prices with zeros or averages, nor does it remove outliers simply because their values are extreme. Instead, it focuses on the period from 2020 onward, when the number of missing values begins to decrease.


### 01 - Setting & Import Libraries


In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path(r"D:\food-price-project")
RAW_FILE   = BASE / "data" / "bronze" / "IDN_RTFP_mkt_2007_2026-10-05.csv"
SILVER_DIR = BASE / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)

# Decisions taken in 01_data_exploration
START_DATE  = "2020-01-01"
END_DATE    = "2026-09-01"                      # Oct 2026 is not finished
COMMODITIES = ["rice", "oil", "sugar", "eggs", "meat_chicken", "onions"]
ID_COLS     = ["geo_id", "mkt_name", "adm1_name", "adm2_name", "lat", "lon", "price_date"]
OUTLIER_MOM_PCT = 50                            # flag if |month-on-month change| > 50%
JAVA_PROVINCES = {"DKI JAKARTA", "JAWA BARAT", "JAWA TENGAH", "JAWA TIMUR",
                  "BANTEN", "DAERAH ISTIMEWA YOGYAKARTA"}

# Every cleaning step is written to this log -> cleaning_log.csv
log = []
def log_step(step, action, rows_before, rows_after, note=""):
    log.append({"step": step, "action": action, "rows_before": rows_before,
                "rows_after": rows_after, "row_change": rows_after - rows_before, "note": note})

### 02 - Load Data

In [7]:
raw = pd.read_csv(RAW_FILE, low_memory=False)
print("Raw shape:", raw.shape)
log_step(1, "Load raw file", len(raw), len(raw), f"{raw.shape[1]} columns")

Raw shape: (53074, 88)


### 03 - Columns To Be Used


In [8]:
patterns = ["c_{}", "{}", "inflation_{}", "trust_{}"]
cols_used = ID_COLS + [p.format(c) for c in COMMODITIES for p in patterns]

missing = [c for c in cols_used if c not in raw.columns]
assert not missing, f"Missing columns: {missing}"

df = raw[cols_used].copy()
print(f"Columns: {raw.shape[1]} -> {df.shape[1]}")
log_step(2, "Select columns", len(df), len(df), f"{raw.shape[1]} -> {df.shape[1]} columns")

Columns: 88 -> 31


### 04 - Check Unwanted Spaces (TRIM)

In [9]:
text_cols = ["geo_id", "mkt_name", "adm1_name", "adm2_name"]
changed = sum((df[c] != df[c].str.strip()).sum() for c in text_cols)
print("Cells with extra spaces:", changed)

for c in text_cols:
    df[c] = df[c].str.strip()
log_step(3, "Trim whitespace in text columns", len(df), len(df), f"{changed} cells changed")

Cells with extra spaces: 0


### 05 - Excluded Market Average From Area

In [10]:
agg_ids = df.loc[(df["adm1_name"] == "Market Average") | (df["adm2_name"] == "Market Average"), "geo_id"].unique()
before = len(df)
df = df[~df["geo_id"].isin(agg_ids)].copy()

print("Removed geo_id:", list(agg_ids))
print("Markets:", df["geo_id"].nunique(), "| Provinces:", df["adm1_name"].nunique(),
      "| Rows without coordinates:", df["lat"].isna().sum())
log_step(4, "Remove aggregate row 'Market Average'", before, len(df),
         f"{df['geo_id'].nunique()} markets, {df['adm1_name'].nunique()} provinces remain")

Removed geo_id: ['gid_idn_national_average']
Markets: 222 | Provinces: 34 | Rows without coordinates: 0


### 06 - Verifying Data Types

In [11]:
df["price_date"] = pd.to_datetime(df["price_date"], errors="coerce")
print("Unparseable dates:", df["price_date"].isna().sum())
assert df["price_date"].notna().all()

num_cols = [c for c in df.columns if c not in text_cols + ["price_date"]]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
log_step(5, "Cast types (dates, numbers)", len(df), len(df), "price_date -> datetime, others -> numeric")

Unparseable dates: 0


### 07 - Unpivot Data

In [12]:
parts = []
for com in COMMODITIES:
    p = df[ID_COLS].copy()
    p["commodity"]         = com
    p["price"]             = df[f"c_{com}"]
    p["price_observed"]    = df[com]
    p["inflation_yoy_pct"] = df[f"inflation_{com}"]
    p["trust"]             = df[f"trust_{com}"]
    parts.append(p)

before = len(df)
silver = pd.concat(parts, ignore_index=True)
silver["is_observed"] = silver["price_observed"].notna().astype(int)
print("Rows:", before, "->", len(silver))
log_step(6, "Unpivot wide -> long", before, len(silver), f"{len(COMMODITIES)} commodities stacked")

Rows: 52836 -> 317016


### 08 - Remove Duplicates

In [13]:
key = ["geo_id", "price_date", "commodity"]
before = len(silver)
n_dup = silver.duplicated(key).sum()
silver = silver.drop_duplicates(key).copy()
print("Duplicates removed:", n_dup)
log_step(7, "Remove duplicates (geo_id, price_date, commodity)", before, len(silver), f"{n_dup} duplicates")

Duplicates removed: 0


### 09 - Remove Invalid Prices

In [14]:
silver["is_invalid_price"] = (silver["price"].isna() | (silver["price"] <= 0)).astype(int)
n_invalid = int(silver["is_invalid_price"].sum())
print("Invalid prices (null or <= 0):", n_invalid)
log_step(8, "Flag invalid prices (null or <= 0)", len(silver), len(silver), f"{n_invalid} flagged, none deleted")

Invalid prices (null or <= 0): 0


### 10 - Checking Outliers 

In [15]:
# Computed BEFORE cutting the period, so Jan 2020 can still be compared with Dec 2019
silver = silver.sort_values(["geo_id", "commodity", "price_date"]).reset_index(drop=True)
prev = silver.groupby(["geo_id", "commodity"])["price"].shift(1)
silver["mom_pct"] = (silver["price"] / prev - 1) * 100
silver["is_outlier"] = (silver["mom_pct"].abs() > OUTLIER_MOM_PCT).astype(int)
n_out = int(silver["is_outlier"].sum())
print("Outliers flagged:", n_out)
log_step(9, f"Flag outliers (|MoM| > {OUTLIER_MOM_PCT}%)", len(silver), len(silver), f"{n_out} flagged, none deleted")

Outliers flagged: 11


### 11 - Filtering Data from 2020 Onwards

In [16]:
before = len(silver)
silver = silver[(silver["price_date"] >= START_DATE) & (silver["price_date"] <= END_DATE)].copy()
print(silver["price_date"].min().date(), "->", silver["price_date"].max().date())
log_step(10, f"Filter period {START_DATE} to {END_DATE}", before, len(silver),
         "before 2020 almost no observed prices; Oct 2026 not finished")

2020-01-01 -> 2026-09-01


### 12 - Creating Derived Columns

In [17]:
silver["year"]  = silver["price_date"].dt.year
silver["month"] = silver["price_date"].dt.month
silver["island_group"] = np.where(silver["adm1_name"].isin(JAVA_PROVINCES), "Jawa", "Luar Jawa")

java_found = set(silver["adm1_name"]) & JAVA_PROVINCES
assert java_found == JAVA_PROVINCES, f"Java provinces not found: {JAVA_PROVINCES - java_found}"
print(silver.groupby("island_group")["adm1_name"].nunique())
log_step(11, "Add year, month, island_group", len(silver), len(silver), "island_group from a fixed list of 6 Java provinces")

island_group
Jawa          6
Luar Jawa    28
Name: adm1_name, dtype: int64


### 13 - Final Verification

In [18]:
n_mk, n_mo, n_cm = silver["geo_id"].nunique(), silver["price_date"].nunique(), silver["commodity"].nunique()
print(f"{n_mk} markets x {n_mo} months x {n_cm} commodities = {n_mk * n_mo * n_cm:,}  |  rows: {len(silver):,}")
assert n_mk * n_mo * n_cm == len(silver), "Panel is not complete"
assert silver.duplicated(["geo_id", "price_date", "commodity"]).sum() == 0
assert silver["is_invalid_price"].sum() == 0

print("\n% observed price by commodity:")
print(silver.groupby("commodity")["is_observed"].mean().mul(100).round(1))
silver.groupby("commodity")["price"].describe().round(0)

222 markets x 81 months x 6 commodities = 107,892  |  rows: 107,892

% observed price by commodity:
commodity
eggs            89.6
meat_chicken    88.2
oil             89.0
onions          89.4
rice            88.7
sugar           89.6
Name: is_observed, dtype: float64


,count,mean,std,min,25%,50%,75%,max
commodity,,,,,,,,
eggs,17982.0,28022.0,4325.0,17550.0,25000.0,27733.0,30400.0,45000.0
meat_chicken,17982.0,35600.0,5990.0,20370.0,31500.0,35095.0,39221.0,60000.0
oil,17982.0,18978.0,3688.0,11698.0,16177.0,19050.0,21594.0,33445.0
onions,17982.0,37756.0,9185.0,15208.0,31530.0,36782.0,42613.0,80000.0
rice,17982.0,13420.0,2333.0,8625.0,11500.0,13500.0,15000.0,21500.0
sugar,17982.0,15796.0,2274.0,10982.0,14000.0,15429.0,18000.0,22000.0


### 14 - Saving the Cleaned Data

In [19]:
final_cols = ["geo_id", "mkt_name", "adm1_name", "adm2_name", "island_group", "lat", "lon",
              "price_date", "year", "month", "commodity",
              "price", "price_observed", "is_observed", "inflation_yoy_pct", "trust",
              "mom_pct", "is_outlier", "is_invalid_price"]
silver = silver[final_cols]

silver.to_csv(SILVER_DIR / "food_prices_cleaned.csv", index=False)

log_df = pd.DataFrame(log)
log_df.to_csv(SILVER_DIR / "cleaning_log.csv", index=False)

print("Saved:", silver.shape)
log_df

Saved: (107892, 19)


,step,action,rows_before,rows_after,row_change,note
0,1,Load raw file,53074,53074,0,88 columns
1,2,Select columns,53074,53074,0,88 -> 31 columns
2,3,Trim whitespace in text columns,53074,53074,0,0 cells changed
3,4,Remove aggregate row 'Market Average',53074,52836,-238,"222 markets, 34 provinces remain"
4,5,"Cast types (dates, numbers)",52836,52836,0,"price_date -> datetime, others -> numeric"
5,6,Unpivot wide -> long,52836,317016,264180,6 commodities stacked
6,7,"Remove duplicates (geo_id, price_date, commodity)",317016,317016,0,0 duplicates
7,8,Flag invalid prices (null or <= 0),317016,317016,0,"0 flagged, none deleted"
8,9,Flag outliers (|MoM| > 50%),317016,317016,0,"11 flagged, none deleted"
9,10,Filter period 2020-01-01 to 2026-09-01,317016,107892,-209124,before 2020 almost no observed prices; Oct 202...
